# CarePath DARAG - Part 1: Data Prep (CPU runtime)

Run on a **CPU runtime** (Runtime > Change runtime type > CPU) so it costs no GPU
units. The slow step is Gipformer ASR (sherpa-onnx, **CPU-only**) transcribing
ViMedCSS into real GEC pairs. Saves artifacts to Google Drive; **Part 2** restores
them and trains.

In [ ]:
!pip install -q "datasets[audio]" soundfile huggingface_hub sherpa-onnx numpy jiwer evaluate
print("CPU data-prep deps installed (no GPU/torch needed for this notebook).")

In [ ]:
import os

# ── Repo source ─────────────────────────────────────────────────────
# The repo is private. Add a Colab Secret named GITHUB_TOKEN (🔑 icon in the left
# sidebar) holding a classic PAT with the "repo" scope, and enable Notebook access.
try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
    os.environ["CAREPATH_REPO_URL"] = f"https://{_token}@github.com/truong-tt/carepath"
    print("GITHUB_TOKEN loaded from Colab Secrets.")
except Exception:
    # Fallback for non-Colab runtimes: paste your token (do NOT commit this value)
    # os.environ["CAREPATH_REPO_URL"] = "https://YOUR_TOKEN_HERE@github.com/truong-tt/carepath"
    raise RuntimeError(
        "Could not load GITHUB_TOKEN from Colab Secrets.\n"
        "Add it via the 🔑 Secrets panel (left sidebar) and enable Notebook access,\n"
        "or uncomment the fallback line above and paste your token."
    )
# ─────────────────────────────────────────────────────────────

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

_PRUNE_DIRS = {'proc', 'sys', 'dev', 'run', 'snap'}
_SKIP_ROOTS = {'/', '/proc', '/sys', '/dev', '/run'}

def is_carepath_repo(path: Path) -> bool:
    return (path / 'scripts' / 'create_gec_pairs.py').exists() and (path / 'apps' / 'api' / 'carepath').exists()

def find_carepath_repo(base: Path) -> Path | None:
    if is_carepath_repo(base):
        return base
    if not base.exists() or str(base) in _SKIP_ROOTS:
        return None
    # os.walk (not rglob) so we can prune /proc, /sys and skip unreadable dirs.
    for dirpath, dirnames, filenames in os.walk(str(base), followlinks=False, onerror=lambda _: None):
        dirnames[:] = [d for d in dirnames if d not in _PRUNE_DIRS]
        if 'create_gec_pairs.py' in filenames and Path(dirpath).name == 'scripts':
            candidate = Path(dirpath).parent
            if is_carepath_repo(candidate):
                return candidate
    return None

def extract_repo_zip(zip_path: Path, extract_dir: Path = Path('/content/carepath_upload')) -> Path | None:
    if not zip_path.exists():
        return None
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extract_dir)
    return find_carepath_repo(extract_dir)

def git_pull_latest(repo: Path) -> None:
    """Pull the newest commit so code fixes arrive without a manual git pull."""
    if not (repo / '.git').exists():
        return
    url = os.environ.get('CAREPATH_REPO_URL')
    try:
        if url:
            subprocess.run(['git', 'remote', 'set-url', 'origin', url], cwd=repo, check=False)
        out = subprocess.run(['git', 'pull', '--ff-only'], cwd=repo, capture_output=True, text=True)
        print(out.stdout.strip() or out.stderr.strip())
    except Exception as exc:
        print(f'git pull skipped: {exc}')

candidates = [
    Path(os.environ.get('CAREPATH_REPO_DIR', '/content/carepath')),
    Path('/content/carepath'),
    Path('/content/CarePath'),
    Path('/content/drive/MyDrive/carepath'),
    Path.cwd(),
    Path.cwd().parent,
]

zip_candidates = [
    Path(value)
    for value in [
        os.environ.get('CAREPATH_REPO_ZIP'),
        '/content/carepath.zip',
        '/content/CarePath.zip',
        '/content/drive/MyDrive/carepath.zip',
        '/content/drive/MyDrive/CarePath.zip',
    ]
    if value
]

REPO_DIR = next((repo for path in candidates if (repo := find_carepath_repo(path))), None)

if REPO_DIR is None:
    REPO_DIR = next((repo for path in zip_candidates if (repo := extract_repo_zip(path))), None)

if REPO_DIR is None:
    repo_url = os.environ.get('CAREPATH_REPO_URL')
    target_dir = Path(os.environ.get('CAREPATH_REPO_DIR', '/content/carepath'))
    if repo_url:
        subprocess.run(['git', 'clone', repo_url, str(target_dir)], check=True)
        REPO_DIR = find_carepath_repo(target_dir)
    else:
        raise RuntimeError(
            'CarePath repo not found. Set CAREPATH_REPO_DIR, CAREPATH_REPO_ZIP, or CAREPATH_REPO_URL.'
        )

if REPO_DIR is None or not is_carepath_repo(REPO_DIR):
    raise RuntimeError(f'Not a CarePath repo: {REPO_DIR}')

git_pull_latest(Path(REPO_DIR))
print(f'Using CarePath repo: {REPO_DIR}')
%cd {REPO_DIR}

In [ ]:
import os, subprocess, sys, textwrap
from pathlib import Path

_env = {
    **os.environ,
    "PYTHONPATH": str(Path(REPO_DIR) / "apps" / "api"),
    # quiet the HuggingFace dataset/model download bars and logging noise
    "HF_DATASETS_DISABLE_PROGRESS_BAR": "1",
    "DATASETS_VERBOSITY": "error",
    "TOKENIZERS_PARALLELISM": "false",
    "TRANSFORMERS_VERBOSITY": "error",
}

def run_step(label: str, cmd: list, log: str = "/tmp/carepath_step.log") -> None:
    """Run cmd with stderr captured; stream stdout live so Colab shows progress."""
    print(f"⏳ {label} …", flush=True)
    with open(log, "w") as lf:
        proc = subprocess.Popen(
            cmd, env=_env, cwd=REPO_DIR,
            stdout=subprocess.PIPE, stderr=lf, text=True,
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
        proc.wait()
    if proc.returncode == 0:
        print(f"✓  {label}\n")
    else:
        stderr_tail = Path(log).read_text()[-3000:]
        print(f"✗  {label} failed (exit {proc.returncode})\n"
              f"── stderr (last 3 000 chars) ──\n{textwrap.indent(stderr_tail, '  ')}")
        raise SystemExit(proc.returncode)

print("run_step ready.")

In [ ]:
# ── Run size ────────────────────────────────────────────────────────
# Smoke defaults keep a first pass fast. For a real run, raise these
# (e.g. LIMIT_PER_SPLIT=None for the whole split, MAX_STEPS=300) and re-run
# the cells below — every step reads these variables.
LIMIT_PER_SPLIT = None    # whole splits instead of 20 rows
SYNTH_COUNT     = 200     # or keep 50 for a first real pass
SYNTH_TTS_LIMIT = 100
MAX_STEPS       = 300     # was 20

def _limit_args(flag: str, value) -> list:
    """Return [flag, str(value)] or [] when value is None (use the whole split)."""
    return [] if value is None else [flag, str(value)]

print(f"LIMIT_PER_SPLIT={LIMIT_PER_SPLIT}  SYNTH_COUNT={SYNTH_COUNT}  "
      f"SYNTH_TTS_LIMIT={SYNTH_TTS_LIMIT}  MAX_STEPS={MAX_STEPS}")

## Build term datastore + real Gipformer GEC pairs (CPU, the long step)

In [ ]:
run_step(
    "Build term datastore",
    ["python", "scripts/build_term_datastore.py",
     "--dataset", "tensorxt/ViMedCSS",
     *_limit_args("--limit-per-split", LIMIT_PER_SPLIT),
     "--output", "artifacts/retrieval/term_datastore_smoke.json"],
)

run_step(
    "Create Gipformer GEC pairs (sherpa-onnx, CPU)",
    ["python", "scripts/create_gec_pairs.py",
     "--output", "artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl",
     *_limit_args("--limit-per-split", LIMIT_PER_SPLIT),
     "--lexicon", "artifacts/retrieval/term_datastore_smoke.json",
     "--resume"],
)

run_step(
    "Evaluate corrections (raw ASR baseline)",
    ["python", "scripts/evaluate_corrections.py",
     "--input", "artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl",
     "--prediction-column", "raw_asr"],
)

## Verify & save artifacts to Google Drive

In [ ]:
from pathlib import Path

artifacts = Path(REPO_DIR) / 'artifacts'

expected = [
    artifacts / 'retrieval' / 'term_datastore_smoke.json',
    artifacts / 'gec_pairs' / 'vimedcss_gipformer_pairs_smoke.jsonl',
]

print("Artifact check")
all_ok = True
for p in expected:
    if p.exists():
        size = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) if p.is_dir() else p.stat().st_size
        print(f"  OK   {p.relative_to(REPO_DIR)}  ({size:,} bytes)")
    else:
        print(f"  MISS {p.relative_to(REPO_DIR)}")
        all_ok = False
print("ALL PRESENT" if all_ok else "SOME MISSING - check cell outputs above")

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

src  = Path(REPO_DIR) / 'artifacts'
dest = Path('/content/drive/MyDrive/carepath_artifacts')

dest.mkdir(parents=True, exist_ok=True)
shutil.copytree(str(src), str(dest), dirs_exist_ok=True)

print(f"Artifacts saved to Google Drive at: {dest}")
print("Contents:")
for p in sorted(dest.rglob('*')):
    if p.is_file():
        print(f"  {p.relative_to(dest)}")